## **Dashboard Preparation & Business Insight**

**Tujuan**

Menyiapkan dataset dan indikator kinerja utama (KPI) untuk pembuatan dashboard interaktif yang dapat digunakan untuk memantau hasil pembelajaran, aktivitas mahasiswa, dan pola pengunduran diri.

**Tujuan Dashboard**

1. Memantau distribusi hasil akhir pembelajaran.
2. Mengukur tingkat kelulusan dan pengunduran diri.
3. Membandingkan aktivitas pembelajaran berdasarkan hasil akhir.
4. Mengidentifikasi kelompok observasi tanpa aktivitas VLE tercatat.
5. Menyajikan insight untuk mendukung pemantauan keterlibatan mahasiswa.

#### **Mengimpor Library dan Memuat Dataset**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

base_path = Path("/content/drive/MyDrive/Data_Analytics/edtech-learning-analytics")

processed_path = base_path / "data" / "processed"
dashboard_path = base_path / "dashboard"

dashboard_path.mkdir(parents=True, exist_ok=True)

eda_data = pd.read_csv(processed_path / "student_eda.csv")

print("Ukuran dataset:", eda_data.shape)
eda_data.head()


Mounted at /content/drive
Ukuran dataset: (32593, 16)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_clicks,active_days,unique_resources,has_vle_activity
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,40.0,55.0,True
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,1435.0,80.0,84.0,True
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,281.0,12.0,22.0,True
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,2158.0,123.0,82.0,True
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,1034.0,70.0,66.0,True


#### **Validasi Data Dashboard**

In [ ]:
student_key = ["code_module", "code_presentation", "id_student"]

print("Jumlah observasi:", len(eda_data))

print("Duplikasi student key:", eda_data.duplicated(subset=student_key).sum())

print("\nMissing values pada kolom utama:")
print(
    eda_data[
        ["final_result", "total_clicks",
         "active_days", "unique_resources"]
    ].isna().sum()
)

print("\nStatus aktivitas VLE:")
print(eda_data["has_vle_activity"].value_counts())


Jumlah observasi: 32593
Duplikasi student key: 0

Missing values pada kolom utama:
final_result        0
total_clicks        0
active_days         0
unique_resources    0
dtype: int64

Status aktivitas VLE:
has_vle_activity
True     29228
False     3365
Name: count, dtype: int64


#### **Menentukan Indikator Kinerja Utama (KPI) Dashboard**

Lima Key Performance Indicators (KPI) digunakan untuk memberikan gambaran umum mengenai hasil pembelajaran dan tingkat aktivitas mahasiswa.

| KPI | Definisi |
|---|---|
| Total Enrollments | Jumlah observasi mahasiswa-modul-periode. |
| Pass Rate | Persentase observasi dengan hasil akhir Pass atau Distinction. |
| Withdrawal Rate | Persentase observasi dengan hasil akhir Withdrawn. |
| Average Total Clicks | Rata-rata total klik dari seluruh observasi, termasuk yang tidak memiliki aktivitas VLE tercatat. |
| No VLE Activity Rate | Persentase observasi tanpa aktivitas VLE tercatat. |



In [ ]:
# Perhitungan KPI Dashboard
total_enrollments = len(eda_data)

pass_rate = (
    eda_data["final_result"]
    .isin(["Pass", "Distinction"])
    .mean() * 100
)

withdrawal_rate = (
    eda_data["final_result"]
    .eq("Withdrawn")
    .mean() * 100
)

avg_total_clicks = eda_data["total_clicks"].mean()

no_activity_rate = (
    (~eda_data["has_vle_activity"].astype(bool))
    .mean() * 100
)

print("Total Enrollments:", total_enrollments)
print(f"Pass Rate: {pass_rate:.2f}%")
print(f"Withdrawal Rate: {withdrawal_rate:.2f}%")
print(f"Average Total Clicks: {avg_total_clicks:.2f}")
print(f"No VLE Activity Rate: {no_activity_rate:.2f}%")

Total Enrollments: 32593
Pass Rate: 47.20%
Withdrawal Rate: 31.16%
Average Total Clicks: 1215.14
No VLE Activity Rate: 10.32%


In [ ]:
# Menyimpan Ringkasan KPI
kpi_summary = pd.DataFrame({
    "metric": [
        "Total Enrollments",
        "Pass Rate",
        "Withdrawal Rate",
        "Average Total Clicks",
        "No VLE Activity Rate"
    ],
    "value": [
        total_enrollments,
        round(pass_rate, 2),
        round(withdrawal_rate, 2),
        round(avg_total_clicks, 2),
        round(no_activity_rate, 2)
    ]
})

kpi_summary.to_csv(dashboard_path / "kpi_summary.csv", index=False)

kpi_summary

,metric,value
0,Total Enrollments,32593.00
1,Pass Rate,47.20
2,Withdrawal Rate,31.16
3,Average Total Clicks,1215.14
4,No VLE Activity Rate,10.32


#### **Persiapan Dataset Dashboard**

In [ ]:
active_clicks = eda_data.loc[
    eda_data["has_vle_activity"].astype(bool),
    "total_clicks"
]

low_threshold = active_clicks.quantile(0.33)
high_threshold = active_clicks.quantile(0.66)

print("Low threshold:", low_threshold)
print("High threshold:", high_threshold)

Low threshold: 385.0
High threshold: 1290.0


In [ ]:
# Membuat Kategori Keterlibatan
def categorize_engagement(row):
    if not row["has_vle_activity"]:
        return "No Activity"
    elif row["total_clicks"] <= low_threshold:
        return "Low"
    elif row["total_clicks"] <= high_threshold:
        return "Medium"
    else:
        return "High"

eda_data["engagement_category"] = eda_data.apply(
    categorize_engagement,
    axis=1
)

print(eda_data["engagement_category"].value_counts())

engagement_category
High           9928
Medium         9654
Low            9646
No Activity    3365
Name: count, dtype: int64


In [ ]:
# Membuat Kelompok Hasil Pembelajaran
eda_data["outcome_group"] = (
    eda_data["final_result"].map({
        "Distinction": "Successful",
        "Pass": "Successful",
        "Fail": "Unsuccessful",
        "Withdrawn": "Withdrawn"
    })
)

print(eda_data["outcome_group"].value_counts())

outcome_group
Successful      15385
Withdrawn       10156
Unsuccessful     7052
Name: count, dtype: int64


In [ ]:
# Menyimpan Dataset Akhir Dashboard
dashboard_data = eda_data.copy()

dashboard_data.to_csv(dashboard_path / "dashboard_data.csv", index=False)

print("Dashboard dataset berhasil disimpan.")
print("Ukuran dataset:", dashboard_data.shape)

Dashboard dataset berhasil disimpan.
Ukuran dataset: (32593, 18)


## **Dokumentasi**

#### **Sumber Data**

Dataset dashboard disiapkan dari student_eda.csv, yang merupakan hasil penggabungan informasi mahasiswa dan agregasi aktivitas VLE.

Dataset ini digunakan sebagai sumber data utama untuk membangun dashboard EdTech Learning Analytics menggunakan Microsoft Power BI.

#### **Validasi Data**

Validasi dilakukan terhadap jumlah observasi, duplikasi key, missing values pada kolom utama, dan indikator aktivitas VLE.

Validasi bertujuan memastikan bahwa dataset yang digunakan telah sesuai dengan kebutuhan analisis dan visualisasi dashboard.

#### **Definisi KPI**

Lima Key Performance Indicators (KPI) utama disiapkan untuk menggambarkan kondisi hasil pembelajaran dan keterlibatan mahasiswa.

| KPI | Definisi |
|---|---|
| Total Enrollments | Jumlah observasi mahasiswa-modul-periode |
| Pass Rate | Persentase observasi dengan hasil Pass dan Distinction |
| Withdrawal Rate | Persentase observasi dengan hasil Withdrawn |
| Average Total Clicks | Rata-rata total klik seluruh observasi |
| No VLE Activity Rate | Persentase observasi tanpa aktivitas VLE tercatat |

#### **Persiapan Fitur**

Dua kolom tambahan dibuat untuk mendukung visualisasi dan analisis dashboard:

- engagement_category: Mengelompokkan tingkat engagement menjadi No Activity, Low, Medium, dan High.
- outcome_group: Mengelompokkan hasil pembelajaran menjadi Successful, Unsuccessful, dan Withdrawn.

Kategori engagement menggunakan batas persentil berdasarkan distribusi total klik pada observasi dengan aktivitas VLE tercatat.

#### **File Keluaran**

Hasil persiapan dashboard disimpan dalam beberapa file berikut:

- dashboard/kpi_summary.csv
- dashboard/dashboard_data.csv

---

### **Penyempurnaan Dashboard dan Wawasan Bisnis Akhir**

#### **Penyempurnaan Dashboard**

Dashboard EdTech Learning Analytics dikembangkan menggunakan Microsoft Power BI dengan komponen:

- Lima KPI Cards.
- Empat visualisasi utama.
- Dua slicer berdasarkan Module dan Presentation.

Penyempurnaan dashboard dilakukan dengan memperhatikan konsistensi tata letak, keterbacaan informasi, pemilihan warna, dan kejelasan judul visualisasi.

#### **Validasi Dashboard**

Validasi dilakukan dengan membandingkan hasil visualisasi Power BI dengan hasil analisis pada Google Colab.

Hasil validasi KPI:

| KPI | Nilai |
|---|---:|
| Total Enrollments | 32.593 |
| Pass Rate | 47,20% |
| Withdrawal Rate | 31,16% |
| Average Total Clicks | 1.215,14 |
| No VLE Activity Rate | 10,32% |

Validasi distribusi learning outcomes:

| Final Result | Jumlah Observasi |
|---|---:|
| Pass | 12.361 |
| Withdrawn | 10.156 |
| Fail | 7.052 |
| Distinction | 3.024 |
| **Total** | **32.593** |

Validasi observasi tanpa aktivitas VLE:

| Final Result | Jumlah Observasi |
|---|---:|
| Withdrawn | 2.988 |
| Fail | 374 |
| Pass | 3 |
| Distinction | 0 |
| **Total** | **3.365** |

#### **Pengujian Interaktivitas Dashboard**

Pengujian interaktivitas dilakukan menggunakan dua slicer, yaitu:

1. **Module:** Memfilter data berdasarkan kode modul pembelajaran.
2. **Presentation:** Memfilter data berdasarkan periode penyelenggaraan pembelajaran.

Pengujian bertujuan memastikan KPI dan visualisasi dapat menyesuaikan data berdasarkan filter yang dipilih.

Selain itu, filter has_vle_activity = False diterapkan secara khusus pada visualisasi Learning Outcomes Without VLE Activity agar hanya menampilkan observasi tanpa aktivitas VLE tercatat.

#### **Wawasan Bisnis Akhir**

Berdasarkan hasil analisis dan visualisasi dashboard, diperoleh beberapa temuan utama:

**1. Hasil Pembelajaran**

Dari total 32.593 observasi mahasiswa-modul-periode, distribusi hasil pembelajaran adalah:

- Pass: 37,93%.
- Withdrawn: 31,16%.
- Fail: 21,64%.
- Distinction: 9,28%.

Kategori Pass memiliki proporsi terbesar, sedangkan Withdrawn mencapai 31,16%. Hal ini menunjukkan adanya tantangan dalam mempertahankan keberlanjutan partisipasi mahasiswa.

**2. Keterlibatan mahasiswa**

Rata-rata total klik berdasarkan hasil pembelajaran:

| Final Result | Average Total Clicks |
|---|---:|
| Distinction | 2.666,76 |
| Pass | 1.921,81 |
| Fail | 651,85 |
| Withdrawn | 313,95 |

Observasi dengan hasil Distinction memiliki rata-rata aktivitas VLE tertinggi, sedangkan Withdrawn memiliki rata-rata terendah. Hal ini menunjukkan adanya hubungan antara aktivitas VLE dan hasil pembelajaran, tetapi tidak membuktikan hubungan sebab-akibat.

**3. Kategori Keterlibatan**

Distribusi kategori engagement:

| Engagement Category | Jumlah Observasi |
|---|---:|
| High | 9.928 |
| Medium | 9.654 |
| Low | 9.646 |
| No Activity | 3.365 |

Sebanyak 10,32% observasi tidak memiliki aktivitas VLE tercatat.

**4. Tanpa Aktivitas VLE**

Dari 3.365 observasi tanpa aktivitas VLE tercatat:

- Withdrawn: 2.988 observasi (88,80%).
- Fail: 374 observasi (11,11%).
- Pass: 3 observasi (0,09%).
- Distinction: 0 observasi (0%).

Sebagian besar observasi tanpa aktivitas VLE tercatat memiliki hasil akhir Withdrawn.

#### **Rekomendasi Bisnis**

Berdasarkan temuan analisis, beberapa rekomendasi yang dapat diberikan adalah:

1. Melakukan pemantauan aktivitas mahasiswa secara berkala untuk mengidentifikasi potensi withdrawal.
2. Mengevaluasi pola keterlibatan mahasiswa berdasarkan aktivitas pada platform pembelajaran.
3. Mengembangkan mekanisme peringatan dini bagi mahasiswa yang belum menunjukkan aktivitas VLE tercatat.
4. Melakukan tindak lanjut terhadap mahasiswa yang memiliki tingkat engagement rendah.

#### **Keterbatasan Analisis**

Analisis ini memiliki beberapa keterbatasan yang perlu diperhatikan:

1. Jumlah klik pada VLE tidak secara langsung menggambarkan tingkat pemahaman atau kualitas pembelajaran mahasiswa.
2. Tidak adanya aktivitas VLE tercatat tidak selalu berarti mahasiswa tidak melakukan kegiatan belajar.
3. Perbedaan durasi partisipasi dapat memengaruhi jumlah aktivitas yang tercatat.
4. Hubungan antara engagement dan hasil pembelajaran tidak dapat diartikan sebagai hubungan sebab-akibat.
5. Analisis bersifat deskriptif sehingga temuan yang diperoleh digunakan untuk mengidentifikasi pola dan mendukung evaluasi lebih lanjut.

#### **Hasil Akhir Proyek**

Hasil akhir proyek berupa dashboard interaktif EdTech Learning Analytics yang dikembangkan menggunakan Microsoft Power BI.

Dashboard menyajikan informasi mengenai hasil pembelajaran, tingkat engagement mahasiswa, dan observasi tanpa aktivitas VLE tercatat melalui KPI Cards, visualisasi, serta filter interaktif berdasarkan modul dan periode pembelajaran.